# Base de Histórico de Parcelas

**Objetivo desta etapa:** entender a estrutura da base, validar sua qualidade e salvá-la limpa e em formato adequado para a construção da ABT.

**Entrada:** `data/raw/historico_parcelas.parquet`

**Saídas:** `data/processed/base_dividas.parquet` e `data/processed/base_pagamentos.parquet`

**Resumo dos resultados**
- A base não tem granularidade simples: mistura *o que deveria ser pago* (dívida) com *o que foi pago* (pagamentos), e uma parcela pode ter várias linhas.
- Foram removidas 4.976 linhas duplicadas (9.952 linhas idênticas, em pares).
- Não foram encontrados valores negativos, e há poucos ausentes (0,22%). Nenhum valor numérico precisou de correção.
- A base foi separada em duas tabelas, uma por parcela e outra por pagamento.

## Features da Base de Histórico de Parcelas



| Variável                 | Descrição                                                          |
|-------------------------|---------------------------------------------------------------------|
| **id_contrato**         | Identificador do contrato de empréstimo.                            |
| **id_cliente**          | Identificador do cliente.                                           |
| **versao_parcela**      | Versão da parcela dentro do contrato, utilizada quando existem alterações. |
| **numero_parcela**      | Número sequencial da parcela dentro do contrato.                    |
| **data_prevista_pagamento** | Data inicialmente prevista para o pagamento da parcela.          |
| **data_real_pagamento** | Data em que o pagamento foi efetivamente realizado.                 |
| **valor_previsto_parcela** | Valor original previsto para a parcela.                          |
| **valor_pago_parcela**  | Valor efetivamente pago pelo cliente.                               |


## Importação dos Dados e Bibliotecas Utilizadas

In [50]:
import pandas as pd
import numpy as np
import pyarrow as pa
import re

In [51]:
df = pd.read_parquet('../../data/raw/historico_parcelas.parquet')

## Dimensões


A base original tem **1.664.384 linhas e 8 colunas**, cobrindo **122.867 contratos**.

Como a base de parcelas é a maior das quatro, o número de linhas não equivale ao de parcelas: a seção seguinte mostra que uma mesma parcela pode aparecer em mais de uma linha.

In [52]:
df.shape

(1664384, 8)

In [53]:
# Mostrar todas as colunas em ordem alfabética
print(sorted(df.columns))

['data_prevista_pagamento', 'data_real_pagamento', 'id_cliente', 'id_contrato', 'numero_parcela', 'valor_pago_parcela', 'valor_previsto_parcela', 'versao_parcela']


In [54]:
df.head()

,id_contrato,id_cliente,versao_parcela,numero_parcela,data_prevista_pagamento,data_real_pagamento,valor_previsto_parcela,valor_pago_parcela
0,1594684,100193,0.0,56,2021-12-21,2021-12-21,301.86,301.86
1,1995642,134723,1.0,38,2021-08-09,2021-08-04,12949.20,12949.20
2,1720935,176364,1.0,9,2024-03-06,2024-03-04,61192.53,61192.53
3,1439208,154898,1.0,20,2024-08-11,2024-08-07,8851.23,8851.23
4,1640082,172575,1.0,4,2022-11-15,2022-11-10,8720.28,8720.28


## Granularidade


**Pergunta:** o que representa cada linha? A chave candidata é (`id_contrato`, `numero_parcela`, `versao_parcela`).

**Por que investigar:** na ABT, os indicadores de atraso são calculados por parcela. Se a chave não for única, contar linhas conta parcelas em duplicidade, e somar `valor_previsto_parcela` conta a dívida mais de uma vez.

A análise seguiu quatro passos:
1. Medir o tamanho do problema (linhas × parcelas × versões).
2. Separar os tipos de repetição.
3. Descobrir o que muda entre as linhas repetidas.
4. Entender a natureza de cada repetição, para definir a regra de tratamento.

### Visão geral


Foram analisados os dados da base e verificou-se que a chave composta por (id_contrato, numero_parcela, versao_parcela) não garante unicidade: existem mais linhas na base do que combinações dessas colunas. Apesar disso, em mais de 94% das parcelas há apenas uma linha por parcela/versão; os casos de repetição ocorrem por diferentes motivos e serão detalhados e tratados nas próximas etapas da análise.

In [55]:
chave_parcela = ["id_contrato", "numero_parcela"]
chave_completa = chave_parcela + ["versao_parcela"]

print("linhas:", len(df))
print("contratos:", df["id_contrato"].nunique())
print("parcelas (contrato, número):", df.drop_duplicates(chave_parcela).shape[0])
print("parcelas × versão:", df.drop_duplicates(chave_completa).shape[0])
print("linhas 100% idênticas:", df.duplicated(keep=False).sum())

linhas: 1664384
contratos: 122867
parcelas (contrato, número): 1574411
parcelas × versão: 1584439
linhas 100% idênticas: 9952


### Tipos de repetição

Foram identificados dois principais tipos de duplicidades nas parcelas: em cerca de 0,6% dos casos, a mesma parcela aparece mais de uma vez devido à existência de diferentes versões (indicando revisões); já em cerca de 4,9% das parcelas, a mesma versão se repete em múltiplas linhas, o que indica registros múltiplos para o mesmo pagamento. 

Para tratar isso, versões diferentes foram consideradas como revisões legítimas, enquanto repetições de uma mesma versão foram analisadas separadamente para evitar contar o débito ou pagamento mais de uma vez.

In [56]:

g = df.groupby(chave_parcela).agg(
    n_linhas=("versao_parcela", "size"),
    n_versoes=("versao_parcela", "nunique"),
)
print(g["n_linhas"].value_counts().sort_index())
print(g["n_versoes"].value_counts().sort_index())


n_linhas
1    1486848
2      85322
3       2122
4         90
5         11
6         16
7          1
8          1
Name: count, dtype: int64
n_versoes
1    1564388
2      10018
3          5
Name: count, dtype: int64


In [57]:

print("parcelas com >1 versão:", (g["n_versoes"] > 1).sum())
print("parcelas com a MESMA versão repetida:", (g["n_linhas"] > g["n_versoes"]).sum())

parcelas com >1 versão: 10023
parcelas com a MESMA versão repetida: 77682


### O que muda entre as linhas repetidas

Observou-se que, para uma mesma versão da parcela, tanto o valor previsto quanto a data de vencimento permanecem constantes; quando há repetições dentro da mesma versão, elas diferem apenas em data e valor de pagamento, indicando múltiplos registros de pagamentos associados à mesma parcela. Já entre versões diferentes, a principal mudança ocorre no valor previsto, enquanto as demais informações quase não se alteram. 

Em resumo, a base mistura registros referentes à dívida (uma linha por parcela/versão) e registros dos pagamentos realizados (possivelmente várias linhas por parcela no mesmo versão). Para análise, tratamos versões diferentes como revisões legítimas do cronograma e lidamos separadamente com repetições dentro da mesma versão, para evitar contabilizar pagamentos em duplicidade.

In [58]:

campos = ["data_prevista_pagamento", "data_real_pagamento",
          "valor_previsto_parcela", "valor_pago_parcela"]

entre_versoes = df.groupby(chave_parcela)[campos + ["versao_parcela"]].nunique(dropna=False)
print("Entre versões (parcelas em que o campo muda):")
print((entre_versoes[entre_versoes["versao_parcela"] > 1][campos] > 1).sum())

dup_completa = df[df.duplicated(chave_completa, keep=False)]
na_mesma_versao = dup_completa.groupby(chave_completa)[campos].nunique(dropna=False)
print("Dentro da mesma versão (grupos em que o campo muda):")
print((na_mesma_versao > 1).sum())


Entre versões (parcelas em que o campo muda):
data_prevista_pagamento       31
data_real_pagamento          187
valor_previsto_parcela     10018
valor_pago_parcela           266
dtype: int64
Dentro da mesma versão (grupos em que o campo muda):
data_prevista_pagamento        0
data_real_pagamento        68537
valor_previsto_parcela         0
valor_pago_parcela         72941
dtype: int64


### Natureza da versão

A análise sugere que o campo `versao_parcela` indica versões do cronograma dos contratos, pois enquanto muitos contratos têm mais de uma versão, quase todas as parcelas aparecem em apenas uma versão. 

Nota-se também uma grande concentração de linhas na versão 0, cuja razão ainda não foi investigada. Nos exemplos verificados, observou-se principalmente três situações: pagamentos parciais legítimos, registros cujos pagamentos juntos correspondem ao valor previsto, e duplicidades, às vezes com pequenas diferenças nos valores. 

Em casos de pagamentos duplicados ou inconsistências evidentes, as duplicidades foram removidas para evitar contabilização em dobro; já os pagamentos parciais foram mantidos por representarem registros legítimos.

In [59]:

print(df.groupby("id_contrato")["versao_parcela"].nunique().value_counts().sort_index().head(10))
print(df["versao_parcela"].value_counts().sort_index().head(10))

versao_parcela
1     65207
2     50522
3      3571
4      1670
5       654
6       359
7       251
8       120
9       130
10       56
Name: count, dtype: int64
versao_parcela
0.0    559191
1.0    989981
2.0     67912
3.0     26654
4.0      6473
5.0      5432
6.0      1844
7.0      1963
8.0       896
9.0       920
Name: count, dtype: int64


In [60]:
# 6. Exemplos reais
ids = g[g["n_linhas"] > 1].head(3).index
for i in ids:
    print(df[(df["id_contrato"] == i[0]) & (df["numero_parcela"] == i[1])].sort_values("versao_parcela"))

         id_contrato  id_cliente  versao_parcela  numero_parcela  \
488508       1000013      215520             1.0              12   
1661246      1000013      215520             1.0              12   

        data_prevista_pagamento data_real_pagamento  valor_previsto_parcela  \
488508               2022-04-18          2022-04-14               13282.695   
1661246              2022-04-18          2022-04-14               13282.695   

         valor_pago_parcela  
488508         13282.695000  
1661246        14292.655321  
        id_contrato  id_cliente  versao_parcela  numero_parcela  \
25016       1000019      176905             1.0               7   
231265      1000019      176905             1.0               7   

       data_prevista_pagamento data_real_pagamento  valor_previsto_parcela  \
25016               2019-04-19          2019-03-20                6072.165   
231265              2019-04-19          2019-04-21                6072.165   

        valor_pago_parcela  
2

### Pagamentos repetidos: parcial, duplicidade ou excesso?

Ao analisar parcelas com múltiplas linhas (na versão mais recente), notou-se que, na maioria dos casos, a soma dos pagamentos era praticamente igual ao valor previsto, indicando que se tratavam em geral de pagamentos parciais legítimos. 

Linhas exatamente idênticas, que resultavam em contabilização duplicada dos pagamentos, foram removidas com `drop_duplicates()`. Essa limpeza eliminou cerca de 5 mil linhas duplicadas, restando pouco mais de 1,65 milhão de registros. 

Após essa etapa, identificou-se que o principal motivo dos excessos residuais eram esses casos de duplicidade exata, mas ainda restaram algumas situações para novas verificações.

In [61]:
chave = ["id_contrato", "numero_parcela"]

ult = df.groupby(chave)["versao_parcela"].transform("max")
u = df[df["versao_parcela"] == ult]
g = u.groupby(chave).agg(prev=("valor_previsto_parcela", "first"),
                         pago=("valor_pago_parcela", "sum"),
                         n=("valor_pago_parcela", "size"))
m = g[(g["n"] > 1) & (g["prev"] > 0)]
print(pd.cut(m["pago"] / m["prev"], [-0.01, 0.5, 0.98, 1.02, 1.5, 1e9]).value_counts(normalize=True).round(3))





(0.98, 1.02]           0.865
(1.5, 1000000000.0]    0.127
(1.02, 1.5]            0.006
(-0.01, 0.5]           0.002
(0.5, 0.98]            0.001
Name: proportion, dtype: float64


In [62]:

ident = df[df.duplicated(keep=False)]
print((ident["valor_pago_parcela"] / ident["valor_previsto_parcela"]).describe())

count    9898.000000
mean        1.006725
std         0.700457
min         0.000009
25%         1.000000
50%         1.000000
75%         1.000000
max        29.375583
dtype: float64


In [63]:

mv = df.groupby(chave).filter(lambda x: x["versao_parcela"].nunique() > 1)
r = mv.groupby(chave)["valor_previsto_parcela"].agg(["min", "max"])
print((r["max"] / r["min"]).describe())

count     10023.000000
mean         41.664291
std        1224.764912
min           1.000000
25%           1.760137
50%           3.263786
75%           7.046256
max      110259.800000
dtype: float64


In [64]:
chave = ["id_contrato", "numero_parcela"]
df = df.drop_duplicates()                      


### Qual versão usar?

Inicialmente foi testado se havia alguma versão de parcela cujo valor previsto "fechava" exatamente com o total pago, mas essa hipótese não se confirmou: em apenas 2% dos casos multiversão isso ocorria, sempre na versão mais antiga. Nos demais casos, o valor pago era maior que o previsto em qualquer versão e o previsto variava bastante entre as versões, indicando que esses valores previstos não são confiáveis ou incorporam outros fatores não identificados. 

Diante disso, optamos por adotar sempre a versão mais recente da parcela, marcando esses casos como `parcela_multiversao`, já que representam menos de 1% das parcelas e a escolha não afeta as datas nem altera indicadores de atraso, apenas reduz a confiança nos valores dessas linhas.

In [ ]:

pag = (df.dropna(subset=["data_real_pagamento"])
         .drop_duplicates(chave + ["data_real_pagamento", "valor_pago_parcela"])
         .groupby(chave)["valor_pago_parcela"].sum().rename("pago_total"))

v = (df.drop_duplicates(chave + ["versao_parcela"])
       [chave + ["versao_parcela", "valor_previsto_parcela"]]
       .merge(pag, on=chave, how="left"))
v["bate"] = (v["pago_total"] / v["valor_previsto_parcela"]).between(0.98, 1.02)

multi = v.groupby(chave)["versao_parcela"].transform("nunique") > 1
m = v[multi].copy()
m["rank_versao"] = m.groupby(chave)["versao_parcela"].rank(method="first")   
print(m.groupby("rank_versao")["bate"].mean().round(3))   
print(m.groupby(chave)["bate"].sum().value_counts())       

rank_versao
1.0    0.02
2.0    0.00
3.0    0.00
Name: bate, dtype: float64
bate
0    9818
1     205
Name: count, dtype: int64


### Excesso de pagamento (soma acima de 1,5× o previsto)

Foram identificadas cerca de 5.124 parcelas com valor pago total aproximadamente o dobro do valor previsto, quase todas formadas por exatamente duas linhas. Ao analisar esses casos, verificou-se que geralmente cada linha individual paga algo próximo do valor previsto, as datas costumam coincidir e frequentemente pelo menos uma das linhas registra exatamente o valor previsto. 

Isso indica duplicidade de registro, provavelmente uma parcela correta acompanhada de uma cópia com valor ligeiramente diferente, diferente de casos de excesso por encargos de atraso (que normalmente aparecem como uma única linha). Para lidar com essas duplicidades, é importante filtrar ou ajustar esses valores ao consolidar os pagamentos.

In [66]:

u = df.copy()
ult = u.groupby(chave)["versao_parcela"].transform("max")
u = u[u["versao_parcela"] == ult]
g = u.groupby(chave).agg(prev=("valor_previsto_parcela", "first"),
                         pago=("valor_pago_parcela", "sum"),
                         n=("valor_pago_parcela", "size"))
m2 = g[(g["n"] > 1) & (g["prev"] > 0)]
print(pd.cut(m2["pago"] / m2["prev"], [-0.01, 0.5, 0.98, 1.02, 1.5, 1e9]).value_counts(normalize=True).round(3))



(0.98, 1.02]           0.922
(1.5, 1000000000.0]    0.070
(1.02, 1.5]            0.005
(-0.01, 0.5]           0.002
(0.5, 0.98]            0.001
Name: proportion, dtype: float64


In [67]:

exc = m2[m2["pago"] / m2["prev"] > 1.5]
print((exc["pago"] / exc["prev"]).describe())
print(exc["n"].value_counts())

count    5124.000000
mean        2.911465
std        34.962230
min         1.517261
25%         1.912945
50%         2.040619
75%         2.157324
max      2363.782252
dtype: float64
n
2    4897
3     220
4       7
Name: count, dtype: int64


In [68]:
chave = ["id_contrato", "numero_parcela"]

mv = df[df.groupby(chave)["versao_parcela"].transform("nunique") > 1]

sem_pag = mv.groupby(chave)["data_real_pagamento"].apply(lambda s: s.notna().sum() == 0)
print("sem nenhum pagamento:", sem_pag.mean().round(3))


sem nenhum pagamento: 0.0


In [69]:

pag = (mv.dropna(subset=["data_real_pagamento"])
         .drop_duplicates(chave + ["data_real_pagamento", "valor_pago_parcela"])
         .groupby(chave)["valor_pago_parcela"].sum().rename("pago_total"))
v = (mv.drop_duplicates(chave + ["versao_parcela"])
       [chave + ["versao_parcela", "valor_previsto_parcela"]]
       .merge(pag, on=chave, how="inner"))
v["rank_versao"] = v.groupby(chave)["versao_parcela"].rank(method="first")
v["razao"] = v["pago_total"] / v["valor_previsto_parcela"]
print(v.groupby("rank_versao")["razao"].describe())

               count       mean          std       min       25%       50%  \
rank_versao                                                                  
1.0          10021.0   4.887725     4.949312  1.000009  1.677131  2.979371   
2.0          10021.0  40.031298  1229.034121  1.011063  1.202190  1.527934   
3.0              5.0   9.433463     6.481109  1.676121  6.635268  6.798080   

                   75%            max  
rank_versao                            
1.0           6.093123      91.390557  
2.0           2.578138  110260.800000  
3.0          14.287170      17.770675  


In [ ]:
u = df.copy()
ult = u.groupby(chave)["versao_parcela"].transform("max")
u = u[u["versao_parcela"] == ult]
g = u.groupby(chave).agg(prev=("valor_previsto_parcela", "first"),
                         pago=("valor_pago_parcela", "sum"),
                         n=("valor_pago_parcela", "size")).reset_index()

exc_df = g[(g["n"] == 2) & (g["prev"] > 0) & (g["pago"] / g["prev"] > 1.5)][chave]
e = u.merge(exc_df, on=chave)
e["razao_linha"] = e["valor_pago_parcela"] / e["valor_previsto_parcela"]
print(e["razao_linha"].describe())                                  
print("mesma data nas 2 linhas:",
      (e.groupby(chave)["data_real_pagamento"].nunique() == 1).mean().round(3))
print("alguma linha igual ao previsto:",
      e.groupby(chave)["razao_linha"].apply(lambda s: (s.sub(1).abs() < 0.005).any()).mean().round(3))

count    9794.000000
mean        1.477100
std        17.926419
min         0.000015
25%         1.000000
50%         1.000000
75%         1.057346
max      1253.285014
Name: razao_linha, dtype: float64
mesma data nas 2 linhas: 0.917
alguma linha igual ao previsto: 0.886


## Tipos

As datas estão como `object` e foram convertidas para `datetime` . `versao_parcela` é `float64` e foi convertido para Int

In [71]:
df.dtypes

id_contrato                  int64
id_cliente                   int64
versao_parcela             float64
numero_parcela               int64
data_prevista_pagamento     object
data_real_pagamento         object
valor_previsto_parcela     float64
valor_pago_parcela         float64
dtype: object

In [72]:

df["versao_parcela"] = df["versao_parcela"].astype(int)

df["data_prevista_pagamento"] = pd.to_datetime(df["data_real_pagamento"], errors="coerce")

df["data_real_pagamento"] = pd.to_datetime(df["data_real_pagamento"], errors="coerce")

## Variaveis Numericas

Após remover as duplicatas exatas, analisamos as variáveis numéricas principais. 

Observamos que a maioria das parcelas tem `versao_parcela` igual a 1, mas há alguns casos de muitas revisões. O `numero_parcela` chega a valores bem altos, possivelmente por renegociações, e os valores previstos e pagos apresentam cauda longa, especialmente nos máximos. 

Não existem valores negativos e as medianas estão de acordo com o esperado para o contexto de empréstimos. Notamos também que comparar o valor pago com o previsto por linha é incorreto, já que pagamentos podem ser parcelados em várias linhas; é necessário fazer essa comparação somando todos os pagamentos de cada parcela.

In [73]:
numerical_cols = [col for col in df.select_dtypes(include='number').columns if col not in ['id_contrato', 'id_cliente']]
df[numerical_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
versao_parcela,1659408.0,0.808429,1.020685,0.0,0.000,1.00,1.0000,5.200000e+01
numero_parcela,1659408.0,14.227380,19.411976,1.0,4.000,8.00,16.0000,1.940000e+02
valor_previsto_parcela,1659408.0,17197.610176,48839.851776,0.0,3973.905,8841.42,17295.3900,2.784334e+06
valor_pago_parcela,1655783.0,17535.763794,53920.516827,0.0,3375.000,8186.13,16726.4325,6.556919e+06


## Valores Ausentes


Só duas colunas têm nulos, ambas com **0,22%**: `data_real_pagamento` e `valor_pago_parcela`. As demais não têm nulos.

**Interpretação:** o nulo em `data_real_pagamento` significa que a parcela não teve pagamento registrado. Isso **não é um erro a corrigir**: é informação. Essas linhas **não** foram removidas nem imputadas. Ficam na tabela de dívida e não entram na de pagamentos, e na ABT contam como parcela vencida em aberto até a `data_ref`.

**Por que o percentual é baixo:** o roll rate será baseado em **atraso de pagamento**, e a base registra sobretudo parcelas que acabaram sendo pagas. As parcelas nunca pagas (3.331) são poucas, mas são os atrasos mais longos da base.

In [74]:
missing_percent = df.isnull().mean() * 100
print("Valores ausentes (%):")
print((missing_percent[missing_percent > 0].sort_values(ascending=False)
       .map(lambda x: f"{x:.2f}%")))

Valores ausentes (%):
data_prevista_pagamento    0.22%
data_real_pagamento        0.22%
valor_pago_parcela         0.22%
dtype: object


## Divisão em duas tabelas e salvamento

A base original mistura informações do previsto e do pago em cada linha, o que dificulta análises corretas, pois parcelas com múltiplos pagamentos aparecem mais de uma vez. 

Para resolver, separamos a base em duas: uma tabela resume cada parcela com sua situação final (valor previsto, última versão, flag de multiversão, indicação se o valor previsto é inválido) e outra lista cada evento de pagamento separadamente (eliminando repetições, calculando o acumulado por parcela). Identificamos também, para cada parcela, quando ela foi considerada quitada (acúmulo ≥ 99% do previsto) e calculamos eventuais excessos de pagamento. Assim, garantimos que cada tipo de informação fica na granularidade correta para as análises.

In [83]:

chave = ["id_contrato", "numero_parcela"]

for c in ["data_prevista_pagamento", "data_real_pagamento"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

# ---------------------------------------------------------
# Tabela 1: DÍVIDA (uma linha por parcela)
# ---------------------------------------------------------
df = df.sort_values(chave + ["versao_parcela"])

n_versoes = df.groupby(chave)["versao_parcela"].transform("nunique")
df["parcela_multiversao"] = (n_versoes > 1).astype(int)

divida = (df.groupby(chave, as_index=False)
            .agg(id_cliente=("id_cliente", "last"),
                 versao_parcela=("versao_parcela", "last"),
                 data_prevista_pagamento=("data_prevista_pagamento", "last"),
                 valor_previsto_parcela=("valor_previsto_parcela", "last"),
                 parcela_multiversao=("parcela_multiversao", "max")))

divida["previsto_invalido"] = (divida["valor_previsto_parcela"] <= 0).astype(int)

# ---------------------------------------------------------
# Tabela 2: PAGAMENTOS (uma linha por evento de pagamento)
# ---------------------------------------------------------
pagamentos = (df.dropna(subset=["data_real_pagamento"])
                [chave + ["data_real_pagamento", "valor_pago_parcela"]]
                .drop_duplicates()          # o mesmo pagamento repetido entre versões
                .sort_values(chave + ["data_real_pagamento"])
                .reset_index(drop=True))

pagamentos["pago_acum"] = pagamentos.groupby(chave)["valor_pago_parcela"].cumsum()

# ---------------------------------------------------------
# Resumo por parcela: quitação e excesso (volta para a dívida)
# ---------------------------------------------------------
p = pagamentos.merge(divida[chave + ["valor_previsto_parcela"]], on=chave, how="left")
valido = p["valor_previsto_parcela"] > 0

# data de quitação: primeiro dia em que o acumulado alcança ~99% do previsto
quit_ = (p[valido & (p["pago_acum"] >= 0.99 * p["valor_previsto_parcela"])]
           .groupby(chave)["data_real_pagamento"].min().rename("data_quitacao"))

# excesso: pago total acima do previsto (duplicidade com ruído / pagamento em dobro)
tot = pagamentos.groupby(chave)["valor_pago_parcela"].sum().rename("pago_total")

divida = (divida.merge(quit_, on=chave, how="left")
                .merge(tot, on=chave, how="left"))
divida["pago_com_encargo"] = (
    (divida["pago_total"] > 1.02 * divida["valor_previsto_parcela"])
    & (divida["valor_previsto_parcela"] > 0)).astype(int)

# versão do pagamento limitada ao previsto (para as features de valor)
pagamentos = pagamentos.merge(divida[chave + ["valor_previsto_parcela"]], on=chave, how="left")
pagamentos["pago_acum_cap"] = np.minimum(pagamentos["pago_acum"], pagamentos["valor_previsto_parcela"])

divida["encargo"] = (divida["pago_total"] - divida["valor_previsto_parcela"]).clip(lower=0)



In [84]:
pagamentos.to_parquet("../../data/processed/base_pagamentos.parquet", index=False)


divida.to_parquet("../../data/processed/base_dividas.parquet", index=False)

## Dicionário das tabelas geradas


### `dividas.parquet` (1.574.411 linhas)


| Variável | Descrição |
|---|---|
| `id_contrato`, `numero_parcela` | Chave da parcela |
| `id_cliente` | Cliente associado |
| `versao_parcela` | Versão usada (a mais recente) |
| `data_prevista_pagamento` | Vencimento |
| `valor_previsto_parcela` | Valor previsto da versão usada |
| `parcela_multiversao` | 1 se a parcela tem mais de uma versão |
| `previsto_invalido` | 1 se o valor previsto é ≤ 0 |
| `data_quitacao` | Data em que o pago acumulado alcança 99% do previsto |
| `pago_total` | Soma de todos os pagamentos |
| `pago_com_encargo` | 1 se `pago_total` supera 1,02× o previsto |
| `encargo` | `pago_total − valor_previsto`, mínimo 0 |




### `pagamentos.parquet` (1.645.938 linhas)


| Variável | Descrição |
|---|---|
| `id_contrato`, `numero_parcela` | Chave da parcela |
| `data_real_pagamento` | Data do pagamento |
| `valor_pago_parcela` | Valor do pagamento |
| `pago_acum` | Acumulado de pagamentos da parcela, em ordem de data |
| `valor_previsto_parcela` | Valor previsto (para conferência) |
| `pago_acum_cap` | `pago_acum` limitado ao previsto |

> **Atenção:** `data_quitacao`, `pago_total`, `encargo` e `pago_com_encargo` usam **todos** os pagamentos, inclusive os posteriores à `data_ref` de qualquer contrato. Servem para explorar a base e para calcular o **target**. Nas **features**, o atraso deve ser recalculado só com pagamentos até a `data_ref` (a partir de `pagamentos`), para evitar vazamento de dados.